<h1>Predictos de problemas cardiovasculares</h1>

<h2>Imports instalation</h2>

In [2]:
# %pip install numpy torch scipy wfdb scikit-learn

<h2>Imports</h2>

In [3]:
import os
import math
import random
import argparse
import json
import datetime 
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wfdb
from scipy.signal import find_peaks, butter, filtfilt
from scipy.stats import pearsonr
from sklearn.metrics import (f1_score, precision_score, recall_score, confusion_matrix, multilabel_confusion_matrix, classification_report)

<h2>Utilidades</h2>

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
 
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return b, a
 
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=360):
    """Filtra ECG entre 0.5-40 Hz para quitar baseline wander y ruido de alta frecuencia."""
    b, a = butter_bandpass(lowcut, highcut, fs)
    return filtfilt(b, a, signal, axis=0)
 
def zscore_normalize(x: np.ndarray, eps=1e-8):
    """Normaliza por canal: media 0, std 1."""
    mean = x.mean(axis=0)
    std  = x.std(axis=0) + eps
    return (x - mean) / std, mean, std

<h2>Datasets</h2>

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET — MIT-BIH y PTB con split por paciente
# ══════════════════════════════════════════════════════════════════════════════

PATHOLOGY_NAMES = ["bradicardia", "taquicardia", "arritmia", "hipoxemia"]

def compute_pathology_labels(ecg_channel: np.ndarray, fs: int, spo2_mean: float) -> torch.Tensor:
    """Etiquetas multi-hot [bradi, taqui, arritmia, hipoxemia], mismo criterio que main.py."""
    norm = (ecg_channel - ecg_channel.mean()) / (ecg_channel.std() + 1e-8)
    peaks, _ = find_peaks(norm, distance=int(0.33 * fs), height=0.3)
    labels = torch.zeros(4)
    if len(peaks) < 5:
        return labels  # muy poca señal para estimar ritmo

    rr = np.diff(peaks) / fs * 1000  # ms
    hr = 60000 / rr
    hr_mean = float(np.mean(hr))

    if hr_mean < 60:
        labels[0] = 1.0  # bradicardia
    elif hr_mean > 100:
        labels[1] = 1.0  # taquicardia

    cv = (np.std(hr) / (np.mean(hr) + 1e-8)) * 100
    sdnn = np.std(rr)
    if cv > 15 or sdnn < 50:
        labels[2] = 1.0  # arritmia

    if spo2_mean < 95:
        labels[3] = 1.0  # hipoxemia

    return labels 

def extract_rhythm_labels(record_path: str):
    """Lee los cambios de ritmo anotados por el cardiólogo en un registro MIT-BIH."""
    ann = wfdb.rdann(record_path, "atr")
    segments = []
    for sample, note in zip(ann.sample, ann.aux_note):
        note = note.strip().strip("\x00")
        if note:
            segments.append((int(sample), note))
    return segments


def record_has_arrhythmia_annotation(record_path: str) -> Optional[bool]:
    """
    True si el ritmo dominante (por duración) en el registro NO es normal.
    None si el registro no tiene anotaciones de ritmo (p.ej. PTB, o MIT-BIH sin marcas de ritmo).
    """
    try:
        segments = extract_rhythm_labels(record_path)
    except Exception:
        return None
    if not segments:
        return None

    normal_tags = {"(N", "(NSR"}
    total, abnormal = 0, 0
    for i, (start, label) in enumerate(segments):
        end = segments[i + 1][0] if i + 1 < len(segments) else start
        dur = max(end - start, 1)
        total += dur
        if label not in normal_tags:
            abnormal += dur

    return None if total == 0 else (abnormal / total) > 0.5

class RealECGDataset(Dataset):
    """
    Carga registros reales de MIT-BIH y/o PTB.
    
    El PPG se deriva del canal 0 del ECG filtrado y normalizado.
    NOTA: para PPG verdadero usa BIDMCDataset más abajo.
    
    split: 'train', 'val', o 'test'
    seed:  para reproducibilidad del split por paciente
    """
 
    def __init__(
        self,
        root_dir: str,
        split: str = "train",
        T: int = 500,
        train_ratio: float = 0.70,
        val_ratio: float = 0.15,
        seed: int = 42,
        use_augmentation: bool = False,
    ):
        self.T = T
        self.split = split
        self.use_augmentation = use_augmentation and (split == "train")
        self.records: List[str] = []
 
        mit_path = os.path.join(root_dir, "mit-bih-arrhythmia-database-1.0.0")
        ptb_path = os.path.join(root_dir, "ptb-diagnostic-ecg-database-1.0.0")
 
        patient_records: Dict[str, List[str]] = {}
 
        if os.path.isdir(mit_path):
            for f in os.listdir(mit_path):
                if f.endswith(".dat"):
                    rec = f.replace(".dat", "")
                    patient_id = f"mit_{rec}"
                    patient_records.setdefault(patient_id, []).append(
                        os.path.join(mit_path, rec)
                    )
 
        if os.path.isdir(ptb_path):
            for root, _, files in os.walk(ptb_path):
                for f in files:
                    if f.endswith(".dat"):
                        rec = f.replace(".dat", "")
                        patient_id = f"ptb_{os.path.basename(root)}"
                        patient_records.setdefault(patient_id, []).append(
                            os.path.join(root, rec)
                        )
 
        patients = sorted(patient_records.keys())
        rng = random.Random(seed)
        rng.shuffle(patients)
 
        n = len(patients)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
 
        if split == "train":
            selected = patients[:n_train]
        elif split == "val":
            selected = patients[n_train : n_train + n_val]
        else:  # test
            selected = patients[n_train + n_val:]
 
        for p in selected:
            self.records.extend(patient_records[p])
 
        print(f"[{split:5s}] {len(selected)} pacientes → {len(self.records)} registros")
 
    def __len__(self):
        return len(self.records)
 
    def _augment_ppg(self, ppg: torch.Tensor) -> torch.Tensor:
        """Augmentación solo en train: ruido, escalado, time shift."""
        # Ruido gaussiano suave
        ppg = ppg + 0.01 * torch.randn_like(ppg)
        # Escalado de amplitud ±20%
        ppg = ppg * (0.8 + 0.4 * random.random())
        return ppg
 
    def __getitem__(self, idx):
        record_path = self.records[idx]
        try:
            record = wfdb.rdrecord(record_path)
            signal = record.p_signal.astype(np.float32)
            fs = record.fs
        except Exception:
            return self.__getitem__((idx + 1) % len(self))
 
        # Resamplear a 360 Hz si es necesario (PTB = 1000 Hz)
        if fs != 360 and fs > 0:
            from scipy.signal import resample
            target_len = int(len(signal) * 360 / fs)
            signal = resample(signal, target_len, axis=0)
 
        try:
            signal = bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=360)
        except Exception:
            pass

        y_patho = compute_pathology_labels(signal[:, 0], fs=360, spo2_mean=97.0)
 
        # Si el registro tiene anotaciones de ritmo reales (MIT-BIH), usarlas en vez
        # del umbral heurístico de CV/SDNN para la etiqueta de arritmia.
        real_arrhythmia = record_has_arrhythmia_annotation(record_path)
        if real_arrhythmia is not None:
            y_patho[2] = 1.0 if real_arrhythmia else 0.0

        total = signal.shape[0]
        if total >= self.T:
            max_start = total - self.T
            start = random.randint(0, max_start) if self.split == "train" else max_start // 2
            signal = signal[start : start + self.T]
        else:
            pad = np.zeros((self.T - total, signal.shape[1]), dtype=np.float32)
            signal = np.vstack([signal, pad])
 
        signal, _, _ = zscore_normalize(signal)
 
        ecg = torch.tensor(signal, dtype=torch.float32)
        if ecg.shape[1] < 12:
            pad = torch.zeros(self.T, 12 - ecg.shape[1])
            ecg = torch.cat([ecg, pad], dim=1)
        ecg = ecg[:, :12]
 
        ppg = ecg[:, 0].clone()
        dppg = torch.zeros_like(ppg); dppg[1:]  = ppg[1:]  - ppg[:-1]
        ddppg = torch.zeros_like(ppg); ddppg[1:] = dppg[1:] - dppg[:-1]
 
        # SpO2 y perfusion simulados con variación fisiológica realista
        t_linspace = torch.linspace(0, self.T / 360, self.T)
        spo2 = 0.97 + 0.01 * torch.sin(2 * math.pi * 0.08 * t_linspace)
        perf = 0.40 + 0.08 * torch.sin(2 * math.pi * 0.20 * t_linspace)
 
        if self.use_augmentation:
            ppg = self._augment_ppg(ppg)
 
        x = torch.stack([ppg, dppg, ddppg, spo2, perf], dim=-1)   # [T, 5]
        # hipoxemia siempre simulada aquí -> no confiable. bradi/taqui/arritmia sí lo son.
        valid_mask = torch.tensor([True, True, True, False])
        return x.float(), ecg.float(), y_patho, valid_mask
 
 
# ══════════════════════════════════════════════════════════════════════════════
# DATASET BIDMC — PPG REAL sincronizado con ECG
# (descarga en: https://physionet.org/content/bidmc/1.0.0/)
# ══════════════════════════════════════════════════════════════════════════════
class BIDMCDataset(Dataset):
    """
    PPG real (BIDMC) con ventanas solapadas por registro.

    Cambios vs versión anterior:
    - Solo 1 lead real (lead II); n_leads=1 en vez de 12 con 11 canales en cero.
    - Windowing con stride: cada registro genera varias ventanas en vez de 1.
    - Normalización z-score una sola vez por registro (no por ventana), para
      no perder la escala relativa de la señal entre ventanas del mismo registro.
    - Registros sin canal spo2/pleth válido se descartan una sola vez en __init__,
      en vez de reintentarse (y avisar) en cada __getitem__.
    """
    def __init__(self, root_dir: str, split: str = "train", T: int = 500,
                 stride: int = 250, seed: int = 42):
        self.T = T
        self.split = split
        self.fs = 125

        path = Path(root_dir)
        all_records = sorted({f.stem for f in path.glob("bidmc[0-9][0-9].hea")})
        if not all_records:
            raise FileNotFoundError(f"No se encontraron registros bidmcNN.hea en {path}")

        rng = random.Random(seed)
        rng.shuffle(all_records)
        n = len(all_records)
        if split == "train":
            recs = all_records[:int(n * 0.70)]
        elif split == "val":
            recs = all_records[int(n * 0.70): int(n * 0.85)]
        else:
            recs = all_records[int(n * 0.85):]

        self.record_data: Dict[str, Dict] = {}
        self.windows: List[Tuple[str, int]] = []
        n_skipped = 0

        for rec in recs:
            rec_path = str(path / rec)
            data = self._load_and_cache_record(rec_path)
            if data is None:
                n_skipped += 1
                continue
            self.record_data[rec_path] = data
            total = len(data["ppg"])
            if total < T:
                self.windows.append((rec_path, 0))
            else:
                for start in range(0, total - T + 1, stride):
                    self.windows.append((rec_path, start))

        print(f"[BIDMC {split:5s}] {len(self.record_data)} registros validos "
              f"({n_skipped} descartados) -> {len(self.windows)} ventanas "
              f"(T={T}, stride={stride})")

    def _load_and_cache_record(self, rec_path: str) -> Optional[Dict]:
        try:
            record = wfdb.rdrecord(rec_path)
            sig = record.p_signal.astype(np.float32)
            names = [n.lower().strip().rstrip(",") for n in record.sig_name]
            numerics = wfdb.rdrecord(rec_path + "n")
            num_names = [n.lower().strip().rstrip(",") for n in numerics.sig_name]
            if "spo2" not in num_names or "hr" not in num_names:
                return None
            spo2_full = numerics.p_signal[:, num_names.index("spo2")]
            hr_full = numerics.p_signal[:, num_names.index("hr")]
            ppg_idx = next(i for i, n in enumerate(names) if "pleth" in n)
            ecg_idx = next(i for i, n in enumerate(names) if n == "ii")
        except Exception as e:
            print(f"[BIDMC] descartando {rec_path} ({type(e).__name__}: {e})")
            return None

        ppg_raw = sig[:, ppg_idx]
        ecg_raw = sig[:, ecg_idx]

        # Normalizacion z-score una sola vez por registro (no por ventana)
        ppg_norm = (ppg_raw - ppg_raw.mean()) / (ppg_raw.std() + 1e-8)
        ecg_norm = (ecg_raw - ecg_raw.mean()) / (ecg_raw.std() + 1e-8)

        return {
            "ppg": ppg_norm.astype(np.float32),
            "ecg": ecg_norm.astype(np.float32),
            "spo2_full": spo2_full,
            "hr_full": hr_full,
        }

    def __len__(self):
        return len(self.windows)

    def _crop(self, arr: np.ndarray, start: int) -> np.ndarray:
        seg = arr[start:start + self.T]
        if len(seg) < self.T:
            seg = np.concatenate([seg, np.zeros(self.T - len(seg), dtype=np.float32)])
        return seg

    def __getitem__(self, idx):
        rec_path, start = self.windows[idx]
        data = self.record_data[rec_path]

        ppg_seg = self._crop(data["ppg"], start)
        ecg_seg = self._crop(data["ecg"], start)

        t0_s, t1_s = start // self.fs, (start + self.T) // self.fs + 1
        spo2_win = data["spo2_full"][t0_s:t1_s]
        hr_win = data["hr_full"][t0_s:t1_s]
        spo2_win = spo2_win[~np.isnan(spo2_win)]
        hr_win = hr_win[~np.isnan(hr_win)]
        spo2_mean = float(np.mean(spo2_win)) if len(spo2_win) else float("nan")
        hr_mean = float(np.mean(hr_win)) if len(hr_win) else float("nan")

        ppg = torch.tensor(ppg_seg, dtype=torch.float32)
        dppg = torch.zeros_like(ppg); dppg[1:] = ppg[1:] - ppg[:-1]
        ddppg = torch.zeros_like(ppg); ddppg[1:] = dppg[1:] - dppg[:-1]
        spo2_ch = torch.full_like(ppg, (spo2_mean if not math.isnan(spo2_mean) else 97.0) / 100.0)
        perf = torch.ones_like(ppg) * 0.5  # BIDMC no reporta perfusion index; placeholder

        x = torch.stack([ppg, dppg, ddppg, spo2_ch, perf], dim=-1)  # [T, 5]

        # Solo 1 lead real -> ecg_t es [T, 1], no [T, 12] con 11 canales en cero
        ecg_t = torch.tensor(ecg_seg, dtype=torch.float32).unsqueeze(-1)  # [T, 1]

        y_patho = torch.zeros(4)
        valid_mask = torch.zeros(4, dtype=torch.bool)
        if not math.isnan(hr_mean):
            if hr_mean < 60:
                y_patho[0] = 1.0
            elif hr_mean > 100:
                y_patho[1] = 1.0
            valid_mask[0] = True
            valid_mask[1] = True
        if not math.isnan(spo2_mean):
            y_patho[3] = 1.0 if spo2_mean < 95.0 else 0.0
            valid_mask[3] = True

        return x, ecg_t, y_patho, valid_mask

<h2>Markdown</h2>

In [6]:
def build_mlp(in_dim, hidden_dims, out_dim, activation=nn.SiLU, dropout=0.0, final_activation=None):
    layers = []
    prev = in_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), activation()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    if final_activation is not None:
        layers.append(final_activation)
    return nn.Sequential(*layers)
 
def scatter_sum(src, index, dim_size):
    B, E, H = src.shape
    out = src.new_zeros(B, dim_size, H)
    for b in range(B):
        out[b].index_add_(0, index, src[b])
    return out
 
@dataclass
class HeartGraphSpec:
    node_names: List[str]
    node_types: List[int]
    chambers: List[int]
    coords: torch.Tensor
    edge_index: torch.Tensor
    edge_attr: torch.Tensor
 
 
def build_default_heart_graph(device="cpu") -> HeartGraphSpec:
    node_names = [
        "SA", "AV", "HIS", "LBB", "RBB",
        "LP1", "LP2", "LP3", "RP1", "RP2",
        "LA_ant", "LA_post", "RA_ant", "RA_post",
        "LV_septal", "LV_anterior", "LV_lateral", "LV_posterior", "LV_apex", "LV_base",
        "RV_septal", "RV_freewall", "RV_apex", "RV_base",
    ]
    type_map = {
        "SA": 0, "AV": 1, "HIS": 2, "LBB": 2, "RBB": 2,
        "LP1": 3, "LP2": 3, "LP3": 3, "RP1": 3, "RP2": 3,
        "LA_ant": 4, "LA_post": 4, "RA_ant": 4, "RA_post": 4,
        "LV_septal": 5, "LV_anterior": 5, "LV_lateral": 5, "LV_posterior": 5, "LV_apex": 5, "LV_base": 5,
        "RV_septal": 5, "RV_freewall": 5, "RV_apex": 5, "RV_base": 5,
    }
    chamber_map = {
        "SA": 0, "AV": 0, "HIS": 0, "LBB": 5, "RBB": 5,
        "LP1": 5, "LP2": 5, "LP3": 5, "RP1": 5, "RP2": 5,
        "LA_ant": 1, "LA_post": 1, "RA_ant": 2, "RA_post": 2,
        "LV_septal": 3, "LV_anterior": 3, "LV_lateral": 3, "LV_posterior": 3, "LV_apex": 3, "LV_base": 3,
        "RV_septal": 4, "RV_freewall": 4, "RV_apex": 4, "RV_base": 4,
    }
    coords_dict = {
        "SA": (-0.2, 0.7, 0.2), "AV": (-0.1, 0.3, 0.0), "HIS": (0.0, 0.1, 0.0),
        "LBB": (-0.2, -0.1, -0.1), "RBB": (0.2, -0.1, -0.1),
        "LP1": (-0.35, -0.35, -0.1), "LP2": (-0.45, -0.45, -0.2), "LP3": (-0.25, -0.55, -0.25),
        "RP1": (0.35, -0.35, -0.1), "RP2": (0.25, -0.55, -0.2),
        "LA_ant": (-0.15, 0.55, 0.1), "LA_post": (-0.10, 0.45, -0.1),
        "RA_ant": (0.15, 0.55, 0.1), "RA_post": (0.10, 0.45, -0.1),
        "LV_septal": (-0.10, -0.30, 0.0), "LV_anterior": (-0.20, -0.35, 0.15),
        "LV_lateral": (-0.45, -0.35, 0.0), "LV_posterior": (-0.25, -0.40, -0.2),
        "LV_apex": (-0.20, -0.65, -0.15), "LV_base": (-0.15, -0.10, 0.0),
        "RV_septal": (0.08, -0.28, 0.0), "RV_freewall": (0.32, -0.35, 0.0),
        "RV_apex": (0.20, -0.58, -0.12), "RV_base": (0.12, -0.12, 0.0),
    }
    name_to_idx = {n: i for i, n in enumerate(node_names)}
 
    def e(a, b, delay, weight, kind):
        return (name_to_idx[a], name_to_idx[b], [delay, weight, kind])
 
    edges = [
        e("SA", "RA_ant", 0.020, 1.0, 0), e("SA", "LA_ant", 0.030, 0.9, 0),
        e("SA", "AV", 0.050, 1.0, 0), e("AV", "HIS", 0.080, 1.0, 0),
        e("HIS", "LBB", 0.015, 1.0, 0), e("HIS", "RBB", 0.015, 1.0, 0),
        e("LBB", "LP1", 0.010, 1.0, 0), e("LBB", "LP2", 0.012, 0.9, 0), e("LBB", "LP3", 0.014, 0.9, 0),
        e("RBB", "RP1", 0.010, 1.0, 0), e("RBB", "RP2", 0.013, 0.9, 0),
    ]
    lv_nodes = ["LV_septal", "LV_anterior", "LV_lateral", "LV_posterior", "LV_apex", "LV_base"]
    rv_nodes = ["RV_septal", "RV_freewall", "RV_apex", "RV_base"]
    atria_l  = ["LA_ant", "LA_post"]
    atria_r  = ["RA_ant", "RA_post"]
    for n in lv_nodes:
        edges += [e("LP1", n, 0.010, 0.8, 0), e("LP2", n, 0.012, 0.7, 0)]
    for n in rv_nodes:
        edges += [e("RP1", n, 0.010, 0.8, 0), e("RP2", n, 0.012, 0.7, 0)]
    for n in atria_l:
        edges += [e("LA_ant", n, 0.020, 0.7, 1), e(n, "LA_ant", 0.020, 0.7, 1)]
    for n in atria_r:
        edges += [e("RA_ant", n, 0.020, 0.7, 1), e(n, "RA_ant", 0.020, 0.7, 1)]
    for a, b in zip(lv_nodes, lv_nodes[1:]):
        edges += [e(a, b, 0.025, 0.7, 1), e(b, a, 0.025, 0.7, 1)]
    for a, b in zip(rv_nodes, rv_nodes[1:]):
        edges += [e(a, b, 0.040, 0.5, 2), e(b, a, 0.040, 0.5, 2)]
 
    src = torch.tensor([x[0] for x in edges], dtype=torch.long, device=device)
    dst = torch.tensor([x[1] for x in edges], dtype=torch.long, device=device)
    edge_attr = torch.tensor([x[2] for x in edges], dtype=torch.float32, device=device)
    edge_index = torch.stack([src, dst], dim=0)
 
    return HeartGraphSpec(
        node_names=node_names,
        node_types=[type_map[n] for n in node_names],
        chambers=[chamber_map[n] for n in node_names],
        coords=torch.tensor([coords_dict[n] for n in node_names], dtype=torch.float32, device=device),
        edge_index=edge_index,
        edge_attr=edge_attr,
    )
 
 
class GraphGRUCell(nn.Module):
    def __init__(self, hidden_dim, edge_dim, msg_dim, dropout=0.0):
        super().__init__()
        self.msg_net = build_mlp(2 * hidden_dim + edge_dim, [msg_dim], msg_dim, dropout=dropout)
        self.gru = nn.GRUCell(msg_dim + hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
 
    def forward(self, h, local_drive, edge_index, edge_attr):
        src_idx, dst_idx = edge_index
        B, N, H = h.shape
        h_src = h[:, src_idx, :]
        h_dst = h[:, dst_idx, :]
        ea = edge_attr.unsqueeze(0).expand(B, -1, -1)
        msgs = self.msg_net(torch.cat([h_src, h_dst, ea], dim=-1))
        agg = scatter_sum(msgs, dst_idx, dim_size=N)
        inp = torch.cat([agg, local_drive], dim=-1)
        h_flat = h.reshape(B * N, H)
        inp_flat = inp.reshape(B * N, -1)
        h_new = self.gru(inp_flat, h_flat).reshape(B, N, H)
        return self.norm(h_new)
 
 
class ECGProjectionHead(nn.Module):
    def __init__(self, hidden_dim, n_leads, source_dim=3):
        super().__init__()
        self.node_to_lead = build_mlp(hidden_dim + source_dim, [hidden_dim], n_leads)
        self.source_head = build_mlp(hidden_dim, [hidden_dim // 2], source_dim)
 
    def forward(self, h, coords):
        B, N, H = h.shape
        coord_exp = coords.unsqueeze(0).expand(B, -1, -1)
        feat = torch.cat([h, coord_exp], dim=-1)
        per_node = self.node_to_lead(feat)
        ecg_t = per_node.mean(dim=1)
        source_t = self.source_head(h).mean(dim=1)
        return ecg_t, source_t
 
 
class OximetryEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, hidden_dim // 2, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(hidden_dim // 2, hidden_dim, kernel_size=5, padding=2)
        self.norm1 = nn.BatchNorm1d(hidden_dim // 2)
        self.norm2 = nn.BatchNorm1d(hidden_dim)
        self.proj = nn.Linear(hidden_dim, hidden_dim)
 
    def forward(self, x):
        z = x.transpose(1, 2)
        z = F.silu(self.norm1(self.conv1(z)))
        z = F.silu(self.norm2(self.conv2(z)))
        z = z.transpose(1, 2)
        return self.proj(z)


class PathologyHead(nn.Module):
    def __init__(self, hidden_dim, n_pathologies=4, dropout=0.1):
        super().__init__()
        self.mlp = build_mlp(hidden_dim, [hidden_dim, hidden_dim // 2], n_pathologies, dropout=dropout)

    def forward(self, h_seq):
        pooled = h_seq.mean(dim=tuple(range(1, h_seq.dim() - 1)))
        return self.mlp(pooled) 


class BioMEMOximetryToECG(nn.Module):
    def __init__(self, graph, input_dim=5, hidden_dim=128, node_emb_dim=32,
                 type_vocab=6, chamber_vocab=6, edge_dim=3, msg_dim=128,
                 n_leads=12, n_layers=2, dropout=0.0):
        super().__init__()
        self.graph = graph
        self.n_nodes = len(graph.node_names)
        self.hidden_dim = hidden_dim
        self.n_leads = n_leads  
 
        self.type_emb = nn.Embedding(type_vocab, node_emb_dim)
        self.chamber_emb = nn.Embedding(chamber_vocab, node_emb_dim)
        self.coord_proj = nn.Linear(3, node_emb_dim)
        self.node_init = build_mlp(3 * node_emb_dim, [hidden_dim, hidden_dim], hidden_dim, dropout=dropout)
        self.oximetry_encoder = OximetryEncoder(input_dim=input_dim, hidden_dim=hidden_dim)
        self.obs_to_nodes = build_mlp(2 * hidden_dim, [hidden_dim], hidden_dim, dropout=dropout)
        self.obs_gate = build_mlp(2 * hidden_dim, [hidden_dim], hidden_dim, dropout=dropout)
        self.cells = nn.ModuleList([
            GraphGRUCell(hidden_dim=hidden_dim, edge_dim=edge_dim, msg_dim=msg_dim, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ion_head = build_mlp(hidden_dim, [hidden_dim], 1)
        self.tension_head = build_mlp(hidden_dim, [hidden_dim], 1)
        self.phase_head = build_mlp(hidden_dim, [hidden_dim], 2)
        self.ecg_head = ECGProjectionHead(hidden_dim=hidden_dim, n_leads=n_leads, source_dim=3)
        self.temporal_refine = nn.GRU(
            input_size=n_leads + 3, hidden_size=hidden_dim, num_layers=1, batch_first=True
        )
        self.out_head = build_mlp(hidden_dim, [hidden_dim], n_leads)
        self.pathology_head = PathologyHead(hidden_dim=hidden_dim, n_pathologies=4, dropout=dropout)
 
        self.register_buffer("node_types", torch.tensor(graph.node_types, dtype=torch.long))
        self.register_buffer("chambers", torch.tensor(graph.chambers, dtype=torch.long))
        self.register_buffer("coords", graph.coords.clone())
        self.register_buffer("edge_index", graph.edge_index.clone())
        self.register_buffer("edge_attr", graph.edge_attr.clone())
 
    def initial_node_state(self, batch_size):
        type_e = self.type_emb(self.node_types)
        ch_e = self.chamber_emb(self.chambers)
        co_e = self.coord_proj(self.coords)
        h0 = self.node_init(torch.cat([type_e, ch_e, co_e], dim=-1))
        return h0.unsqueeze(0).expand(batch_size, -1, -1).contiguous()
 
    def observation_to_nodes(self, obs_t, h):
        obs_node = obs_t.unsqueeze(1).expand(-1, self.n_nodes, -1)
        fused = torch.cat([obs_node, h], dim=-1)
        gate = torch.sigmoid(self.obs_gate(fused))
        return gate * self.obs_to_nodes(fused)
 
    def step(self, h, obs_t):
        local_drive = self.observation_to_nodes(obs_t, h)
        for cell in self.cells:
            h = cell(h, local_drive, self.edge_index, self.edge_attr)
        ecg_t_raw, source_t = self.ecg_head(h, self.coords)
        aux = {
            "vm": self.ion_head(h).squeeze(-1),
            "tension": self.tension_head(h).squeeze(-1),
            "phase": self.phase_head(h),
            "source": source_t,
        }
        return h, ecg_t_raw, aux
 
    def forward(self, x, h0=None, return_aux=True):
        B, T, _ = x.shape
        h = self.initial_node_state(B) if h0 is None else h0
        obs_seq = self.oximetry_encoder(x)
 
        ecg_raw_seq, source_seq, vm_seq, tension_seq, phase_seq = [], [], [], [], []
        node_states_seq = []
        for t in range(T):
            h, ecg_t, aux = self.step(h, obs_seq[:, t, :])
            ecg_raw_seq.append(ecg_t)
            source_seq.append(aux["source"])
            vm_seq.append(aux["vm"])
            tension_seq.append(aux["tension"])
            phase_seq.append(aux["phase"])
            node_states_seq.append(h)
 
        ecg_raw = torch.stack(ecg_raw_seq, dim=1)
        source = torch.stack(source_seq, dim=1)
        z, _ = self.temporal_refine(torch.cat([ecg_raw, source], dim=-1))
        node_states_seq = torch.stack(node_states_seq, dim=1)
        pathology_logits = self.pathology_head(node_states_seq)
        ecg = self.out_head(z) + ecg_raw
 
        out = {"ecg": ecg, "pathology_logits": pathology_logits}
        if return_aux:
            out.update({
                "source": source,
                "vm": torch.stack(vm_seq, dim=1),
                "tension": torch.stack(tension_seq, dim=1),
                "phase": torch.stack(phase_seq, dim=1),
                "final_state": h,
            })
        return out


<h2>Funcion de perdida mejorada</h2>

In [7]:
def masked_bce(logits, targets, valid_mask):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    loss = loss * valid_mask.float()
    denom = valid_mask.float().sum().clamp(min=1.0)
    return loss.sum() / denom

class ImprovedLoss(nn.Module):
    """
    Combina:
      - MSE (forma de onda exacta)
      - Correlación de Pearson (morfología)
      - Suavidad temporal (evita artefactos)
      - Regularización física (vm, tensión, fuentes)
    """
    def __init__(self, mse_w=1.0, pearson_w=0.3, smooth_w=0.05, phys_w=0.05, source_w=0.01, pathology_w=0.2):
        super().__init__()
        self.mse_w = mse_w
        self.pearson_w = pearson_w
        self.smooth_w = smooth_w
        self.phys_w = phys_w
        self.source_w = source_w
        self.pathology_w = pathology_w
        self.bce = nn.BCEWithLogitsLoss()

    def pearson_loss(self, pred, target):
        vp = pred - pred.mean(dim=1, keepdim=True)
        vt = target - target.mean(dim=1, keepdim=True)
        corr = (vp * vt).sum(dim=1) / (
            vp.norm(dim=1) * vt.norm(dim=1) + 1e-8
        )
        return (1.0 - corr).mean()

    def temporal_smoothness(self, y):
        dy = y[:, 1:, :] - y[:, :-1, :]
        return (dy ** 2).mean()

    def physiological_regularization(self, pred):
        reg = 0.1 * (pred["vm"] ** 2).mean()
        reg += 0.1 * F.relu(pred["tension"].abs() - 5.0).mean()
        dsrc = pred["source"][:, 1:, :] - pred["source"][:, :-1, :]
        reg += 0.2 * (dsrc ** 2).mean()
        return reg

    def forward(self, pred, target_ecg, target_pathology=None, valid_mask=None):
        ecg = pred["ecg"]
        l_mse = F.mse_loss(ecg, target_ecg)
        l_pearson = self.pearson_loss(ecg, target_ecg)
        l_smooth = self.temporal_smoothness(ecg)
        l_phys = self.physiological_regularization(pred)
        l_source = (pred["source"] ** 2).mean()

        total = (
            self.mse_w * l_mse
          + self.pearson_w * l_pearson
          + self.smooth_w * l_smooth
          + self.phys_w * l_phys
          + self.source_w * l_source
        )
        losses = {
            "loss": total,
            "mse": l_mse,
            "pearson": l_pearson,
            "smooth": l_smooth,
            "phys": l_phys,
            "source_reg": l_source,
        }

        if target_pathology is not None:
            if valid_mask is not None:
                l_patho = masked_bce(pred["pathology_logits"], target_pathology, valid_mask)
            else:
                l_patho = self.bce(pred["pathology_logits"], target_pathology)
            losses["loss"] = losses["loss"] + self.pathology_w * l_patho
            losses["pathology_bce"] = l_patho

        return losses

<h2>Metricas de evalución</h2>

In [8]:
def compute_metrics(pred_ecg: torch.Tensor, true_ecg: torch.Tensor) -> Dict[str, float]:
    """
    Calcula métricas clínicas relevantes:
      - RMSE por lead
      - PRD (Percent Root-mean-square Difference)
      - Correlación de Pearson promedio
    """
    pred = pred_ecg.detach().cpu().numpy()
    true = true_ecg.detach().cpu().numpy()
 
    rmse_leads, prd_leads, corr_leads = [], [], []
    for lead in range(pred.shape[-1]):
        p = pred[:, :, lead].flatten()
        t = true[:, :, lead].flatten()
        rmse = np.sqrt(np.mean((p - t) ** 2))
        prd = 100.0 * np.sqrt(np.sum((p - t) ** 2) / (np.sum(t ** 2) + 1e-8))
        corr, _ = pearsonr(p, t) if p.std() > 1e-6 and t.std() > 1e-6 else (0.0, 1.0)
        rmse_leads.append(rmse)
        prd_leads.append(prd)
        corr_leads.append(corr)
 
    return {
        "rmse_mean": float(np.mean(rmse_leads)),
        "prd_mean": float(np.mean(prd_leads)),
        "corr_mean": float(np.mean(corr_leads)),
        "prd_per_lead": prd_leads,
        "corr_per_lead": corr_leads,
    }

def compute_pathology_metrics(logits, targets, valid_mask=None, threshold=0.5):
    probs = torch.sigmoid(logits).detach().cpu().numpy()
    preds = (probs >= threshold).astype(int)
    true = targets.detach().cpu().numpy().astype(int)

    mask = valid_mask.detach().cpu().numpy().astype(bool) if valid_mask is not None else np.ones_like(true, dtype=bool)

    per_class = {}
    f1s_for_macro = []

    for i, name in enumerate(PATHOLOGY_NAMES):
        col_mask = mask[:, i]
        n_valid = int(col_mask.sum())

        if n_valid == 0:
            per_class[name] = {"f1": None, "precision": None, "recall": None,
                                "support": 0, "n_valid_samples": 0, "confusion_matrix": None}
            continue  # <- se salta esta clase, no entra al promedio

        y_true_i = true[col_mask, i]   # filtra SOLO las filas válidas de esta columna
        y_pred_i = preds[col_mask, i]

        f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
        precision = precision_score(y_true_i, y_pred_i, zero_division=0)
        recall = recall_score(y_true_i, y_pred_i, zero_division=0)
        cm = confusion_matrix(y_true_i, y_pred_i, labels=[0, 1])

        per_class[name] = {"f1": round(float(f1),4), "precision": round(float(precision),4),
                            "recall": round(float(recall),4), "support": int(y_true_i.sum()),
                            "n_valid_samples": n_valid, "confusion_matrix": cm.tolist()}
        f1s_for_macro.append(f1)

    macro_f1 = float(np.mean(f1s_for_macro)) if f1s_for_macro else None
    return {"macro_f1": round(macro_f1,4) if macro_f1 is not None else None,
            "n_classes_evaluated": len(f1s_for_macro), "per_class": per_class}


def print_pathology_report(metrics: Dict):
    print(f"\n{'Patología':<15}{'F1':>8}{'Precision':>12}{'Recall':>10}{'Support':>10}")
    print("-" * 55)
    for name in PATHOLOGY_NAMES:
        m = metrics["per_class"][name]
        f1        = m.get("f1") or 0.0
        precision = m.get("precision") or 0.0
        recall    = m.get("recall") or 0.0
        support   = m.get("support") or 0
        print(f"{name:<15}{f1:>8.3f}{precision:>12.3f}{recall:>10.3f}{support:>10d}")
    print("-" * 55)
    print(f"{'F1 macro':<15}{metrics['macro_f1']:>8.3f}")

    print("\nMatrices de confusión por patología ([[TN, FP], [FN, TP]]):")
    for name in PATHOLOGY_NAMES:
        cm = metrics["per_class"][name]["confusion_matrix"]
        print(f"  {name:<15}: {cm}")

<h2>Loop de entrenamiento y evaluación</h2>

In [9]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, scaler=None):
    model.train()
    meters = {"loss": 0.0, "mse": 0.0, "pearson": 0.0, "smooth": 0.0}
    n = 0
 
    for x, y, y_patho, valid_mask in loader:
        x, y, y_patho, valid_mask = (
            x.to(device), y.to(device), y_patho.to(device), valid_mask.to(device)
        )
        optimizer.zero_grad()
 
        if scaler is not None:
            # Mixed precision (GPU)
            with torch.cuda.amp.autocast():
                pred = model(x, return_aux=True)
                loss_dict = criterion(pred, y, target_pathology=y_patho, valid_mask=valid_mask)
            scaler.scale(loss_dict["loss"]).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(x, return_aux=True)
            loss_dict = criterion(pred, y, target_pathology=y_patho, valid_mask=valid_mask)
            loss_dict["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
 
        if scheduler is not None:
            scheduler.step()
 
        bs = x.shape[0]
        n += bs
        for k in meters:
            if k in loss_dict:
                meters[k] += loss_dict[k].item() * bs
 
    for k in meters:
        meters[k] /= max(n, 1)
    return meters
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    meters = {"loss": 0.0, "mse": 0.0, "pearson": 0.0, "pathology_bce": 0.0}
    all_pred, all_true = [], []
    all_patho_logits, all_patho_true, all_patho_mask = [], [], []
    n = 0

    for x, y, y_patho, valid_mask in loader:
        x, y, y_patho, valid_mask = (
            x.to(device), y.to(device), y_patho.to(device), valid_mask.to(device)
        )
        pred = model(x, return_aux=True)
        loss_dict = criterion(pred, y, target_pathology=y_patho, valid_mask=valid_mask)

        all_pred.append(pred["ecg"])
        all_true.append(y)
        all_patho_logits.append(pred["pathology_logits"])
        all_patho_true.append(y_patho)
        all_patho_mask.append(valid_mask)

        bs = x.shape[0]
        n += bs
        for k in meters:
            if k in loss_dict:
                meters[k] += loss_dict[k].item() * bs

    for k in meters:
        meters[k] /= max(n, 1)

    # Métricas clínicas sobre todos los batches
    pred_cat = torch.cat(all_pred, dim=0)
    true_cat = torch.cat(all_true, dim=0)
    metrics = compute_metrics(pred_cat, true_cat)
    meters.update(metrics)

    # Métricas de patología sobre todos los batches
    patho_logits_cat = torch.cat(all_patho_logits, dim=0)
    patho_true_cat = torch.cat(all_patho_true, dim=0)
    patho_mask_cat = torch.cat(all_patho_mask, dim=0)
    meters["pathology_metrics"] = compute_pathology_metrics(
        patho_logits_cat, patho_true_cat, valid_mask=patho_mask_cat
    )

    return meters

<h1>Cleaner</h1>

In [10]:
def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

<h2>Pesos de patologias</h2>

In [11]:
def compute_pathology_pos_weight(dataset, cap: float = 6.0) -> torch.Tensor:
    all_y, all_mask = [], []
    for i in range(len(dataset)):
        _, _, y, mask = dataset[i]
        all_y.append(y)
        all_mask.append(mask)
    Y = torch.stack(all_y)      # [N, 4]
    M = torch.stack(all_mask)   # [N, 4]

    pos = (Y * M).sum(dim=0)
    neg = ((1 - Y) * M).sum(dim=0)
    pos_weight = (neg / pos.clamp(min=1)).clamp(max=cap)
    print("pos_weight por patología:", dict(zip(PATHOLOGY_NAMES, pos_weight.tolist())))
    return pos_weight

<h2>Main</h2>

In [12]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", type=str, default=".")
    parser.add_argument("--bidmc_path", type=str, default="bidmc-ppg-and-respiration-dataset-1.0.0")
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--batch_size", type=int, default=16)          # antes 8
    parser.add_argument("--T", type=int, default=500)
    parser.add_argument("--stride", type=int, default=250)             # NUEVO: solapamiento 50%
    parser.add_argument("--hidden_dim", type=int, default=64)          # antes 128
    parser.add_argument("--n_layers", type=int, default=1)             # antes 2
    parser.add_argument("--dropout", type=float, default=0.15)         # NUEVO, antes hardcodeado 0.05
    parser.add_argument("--lr", type=float, default=1e-3)              # antes 3e-3
    parser.add_argument("--patience", type=int, default=15)
    parser.add_argument("--output_dir", type=str, default="checkpointsModelo2_v2")  # NUEVO dir, ver nota abajo
    parser.add_argument("--seed", type=int, default=42)
    args, _ = parser.parse_known_args()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Usando device: {device}")

    if args.bidmc_path:
        print("Usando PPG REAL (BIDMC), 1 lead")
        train_ds = BIDMCDataset(args.bidmc_path, split="train", T=args.T, stride=args.stride, seed=args.seed)
        val_ds = BIDMCDataset(args.bidmc_path, split="val", T=args.T, stride=args.stride, seed=args.seed)
        test_ds = BIDMCDataset(args.bidmc_path, split="test", T=args.T, stride=args.stride, seed=args.seed)
    else:
        print("Usando pseudo-PPG (MIT-BIH / PTB). Para PPG real usa --bidmc_path")
        train_ds = RealECGDataset(args.data_root, split="train", T=args.T, use_augmentation=True, seed=args.seed)
        val_ds = RealECGDataset(args.data_root, split="val", T=args.T, use_augmentation=False, seed=args.seed)
        test_ds = RealECGDataset(args.data_root, split="test", T=args.T, use_augmentation=False, seed=args.seed)

    import platform
    n_workers = 0 if platform.system() == "Windows" else 2
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=n_workers, pin_memory=(n_workers > 0))
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=n_workers, pin_memory=(n_workers > 0))
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, num_workers=n_workers, pin_memory=(n_workers > 0))

    graph = build_default_heart_graph(device=device)
    model = BioMEMOximetryToECG(
        graph=graph, input_dim=5, hidden_dim=args.hidden_dim,
        node_emb_dim=32, msg_dim=128, n_leads=1,               # antes 12
        n_layers=args.n_layers, dropout=args.dropout,
    ).to(device)
    print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=args.lr,
        steps_per_epoch=len(train_loader),
        epochs=args.epochs,
        pct_start=0.25,                                        # antes 0.1
        anneal_strategy="cos",
    )

    scaler = torch.cuda.amp.GradScaler() if device == "cuda" else None
    criterion = ImprovedLoss(mse_w=1.0, pearson_w=0.3, smooth_w=0.05, phys_w=0.05, source_w=0.01)

    checkpoint_path = os.path.join(args.output_dir, "best_model.pt")
    start_epoch = 1
    best_val_loss = float("inf")
    patience_counter = 0
    history = []

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_loss = ckpt["val_loss"]
        history_path = os.path.join(args.output_dir, "history.json")
        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)
        steps_done = (start_epoch - 1) * len(train_loader)
        for _ in range(steps_done):
            scheduler.step()
        print(f"Retomando desde epoch {start_epoch} | mejor val_loss hasta ahora: {best_val_loss:.4f}")
    else:
        print("No se encontró checkpoint — entrenamiento desde cero")

    for epoch in range(start_epoch, args.epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device, scaler)
        va = evaluate(model, val_loader, criterion, device)

        history.append(to_native({"epoch": epoch, "train": tr, "val": va}))
        with open(os.path.join(args.output_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss={tr['loss']:.4f} MSE={tr['mse']:.4f} Pearson={tr['pearson']:.4f} | "
            f"Val Loss={va['loss']:.4f} RMSE={va.get('rmse_mean', 0):.4f} "
            f"Corr={va.get('corr_mean', 0):.4f} PRD={va.get('prd_mean', 0):.2f}% "
            f"| Patho F1={va['pathology_metrics']['macro_f1']:.3f}"
        )

        if va["loss"] < best_val_loss:
            best_val_loss = va["loss"]
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "val_loss": va["loss"],
                "val_corr": va.get("corr_mean", 0),
                "args": vars(args),
            }, os.path.join(args.output_dir, "best_model.pt"))
            print(f"  ✓ Guardado nuevo mejor modelo (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print(f"\nEarly stopping en epoch {epoch} (sin mejora por {args.patience} épocas)")
                break

    print("\n── Evaluación en test set ──")
    ckpt = torch.load(os.path.join(args.output_dir, "best_model.pt"), map_location=device)
    model.load_state_dict(ckpt["model_state"])
    test_metrics = evaluate(model, test_loader, criterion, device)
    print(f"Test RMSE: {test_metrics['rmse_mean']:.4f}")
    print(f"Test Corr: {test_metrics['corr_mean']:.4f}")
    print(f"Test PRD: {test_metrics['prd_mean']:.2f}%")
    print(f"PRD lead II: {test_metrics['prd_per_lead'][0]:.2f}%")   # antes: loop sobre 12 nombres

    patho_metrics = test_metrics["pathology_metrics"]
    print("\n=== Clasificación de patologías (test) ===")
    print_pathology_report(patho_metrics)

    PATHO_LOG_PATH = os.path.join(args.output_dir, "pathology_test_runs.json")
    previous_runs = []
    if os.path.exists(PATHO_LOG_PATH):
        with open(PATHO_LOG_PATH) as f:
            previous_runs = json.load(f)

    current_run = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "epoch": ckpt["epoch"],
        "macro_f1": patho_metrics["macro_f1"],
        "per_class_f1": {name: patho_metrics["per_class"][name]["f1"] for name in PATHOLOGY_NAMES},
    }

    print("\n" + "=" * 60)
    if previous_runs:
        last_run = previous_runs[-1]
        print("COMPARACIÓN PATOLOGÍAS CON LA CORRIDA ANTERIOR")
        print("=" * 60)
        d_f1 = current_run["macro_f1"] - last_run["macro_f1"]
        print(f"{'F1 macro':<15}{last_run['macro_f1']:>12.4f}{current_run['macro_f1']:>12.4f}{d_f1:>+12.4f}")
        for name in PATHOLOGY_NAMES:
            prev = last_run["per_class_f1"].get(name, 0.0)
            curr = current_run["per_class_f1"].get(name, 0.0)
            print(f"  F1 {name:<12}: {prev:.4f} -> {curr:.4f}  ({curr - prev:+.4f})")
        print("\nEl modelo MEJORÓ" if d_f1 > 0 else ("\nEl modelo EMPEORÓ" if d_f1 < 0 else "\nSin cambio"))
    else:
        print("No hay corridas anteriores registradas para patologías. Esta es la primera.")
    print("=" * 60)

    previous_runs.append(current_run)
    with open(PATHO_LOG_PATH, "w") as f:
        json.dump(previous_runs, f, indent=2)


if __name__ == "__main__":
    main()

Usando device: cpu
Usando PPG REAL (BIDMC), 1 lead
[BIDMC train] 36 registros validos (1 descartados) -> 8604 ventanas (T=500, stride=250)
[BIDMC val  ] 8 registros validos (0 descartados) -> 1912 ventanas (T=500, stride=250)
[BIDMC test ] 8 registros validos (0 descartados) -> 1912 ventanas (T=500, stride=250)
Parámetros: 182,125


C:\Users\Sistemas\AppData\Local\Temp\ipykernel_30472\3232624719.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Retomando desde epoch 10 | mejor val_loss hasta ahora: 1.2877
Epoch 010 | Train Loss=1.2859 MSE=0.9800 Pearson=0.8676 | Val Loss=1.3265 RMSE=1.0032 Corr=0.0332 PRD=100.31% | Patho F1=0.221
Epoch 011 | Train Loss=1.1927 MSE=0.9332 Pearson=0.7653 | Val Loss=1.4715 RMSE=1.0563 Corr=-0.0851 PRD=105.62% | Patho F1=0.220
Epoch 012 | Train Loss=1.0612 MSE=0.8490 Pearson=0.6379 | Val Loss=1.4584 RMSE=1.0592 Corr=-0.0401 PRD=105.91% | Patho F1=0.335
Epoch 013 | Train Loss=1.0026 MSE=0.8079 Pearson=0.5858 | Val Loss=1.5426 RMSE=1.0969 Corr=-0.0656 PRD=109.67% | Patho F1=0.312
Epoch 014 | Train Loss=0.9624 MSE=0.7800 Pearson=0.5543 | Val Loss=1.4172 RMSE=1.0526 Corr=0.0463 PRD=105.24% | Patho F1=0.379
Epoch 015 | Train Loss=0.9444 MSE=0.7668 Pearson=0.5402 | Val Loss=1.5272 RMSE=1.0901 Corr=-0.0568 PRD=109.00% | Patho F1=0.337
Epoch 016 | Train Loss=0.9339 MSE=0.7595 Pearson=0.5334 | Val Loss=1.4945 RMSE=1.0803 Corr=-0.0204 PRD=108.01% | Patho F1=0.300
Epoch 017 | Train Loss=0.9197 MSE=0.7489 Pea

KeyboardInterrupt: 